# Explorador de Momento Óptimo con Alertas + Share of Wallet
> **Basado en la Sección 7 de `smart_demand_signals.ipynb`**

Este notebook extiende el explorador interactivo con dos nuevas capas de inteligencia:

1. **Alertas de Intervalo** — el sistema avisa proactivamente cuando:
   - ⚠️ El cliente **entra en su ventana de compra** (alerta anticipación al inicio del intervalo).
   - 🔴 La ventana **cierra sin que el cliente haya comprado** (alerta de cierre al final del intervalo).

2. **Share of Wallet y Velocidad del Share** — detecta el *momentum*:
   - Barras verdes = el cliente nos está comprando más que antes → fidelizar y hacer cross-sell.
   - Barras rojas = el cliente nos está comprando menos → riesgo de fuga a la competencia.


## 0. Setup e importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

REFERENCE_DATE = pd.Timestamp('2025-12-30')
CUTOFF_12M     = REFERENCE_DATE - pd.DateOffset(months=12)

print(f'✅ Setup completado — Fecha de referencia: {REFERENCE_DATE.date()}')


## 1. Carga de datos

In [ ]:
comm = pd.read_csv('data/master_commodities.csv', parse_dates=['Fecha'])
tech = pd.read_csv('data/master_technicals.csv',  parse_dates=['Fecha'])

print(f'Commodities: {comm.shape[0]:,} filas, {comm.shape[1]} columnas')
print(f'Técnicos:    {tech.shape[0]:,} filas, {tech.shape[1]} columnas')
print(f'Rango fechas: {min(comm.Fecha.min(), tech.Fecha.min()).date()} → '
      f'{max(comm.Fecha.max(), tech.Fecha.max()).date()}')
print(f'Clientes únicos (commodities): {comm.Id_Cliente.nunique():,}')


## 2. Preprocesamiento

In [ ]:
# ── Baseline: excluir devoluciones y campañas ────────────────────────────────
comm_base = comm[(comm.es_devolucion == 0) & (comm.en_campana == 0)].copy()
tech_base = tech[(tech.es_devolucion == 0) & (tech.en_campana == 0)].copy()

# ── Agregación mensual (commodities) — necesaria para Share of Wallet ─────────
comm_base['year_month'] = comm_base.Fecha.dt.to_period('M')
monthly = comm_base.groupby(['Id_Cliente', 'Familia_Potencial', 'year_month']).agg(
    euros_venuts  = ('Valores_H',          'sum'),
    num_pedidos   = ('Num.Fact',           'nunique'),
    potencial_eur = ('Potencial_EUR_anual', 'first'),
).reset_index()
monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()

# ── Gaps entre pedidos: Commodities (valid_gaps) ──────────────────────────────
pedidos_comm = (
    comm_base
    .groupby(['Id_Cliente', 'Familia_Potencial', 'Num.Fact'], sort=False)['Fecha']
    .min().reset_index()
    .sort_values(['Id_Cliente', 'Familia_Potencial', 'Fecha'])
)
pedidos_comm['gap_dies'] = (
    pedidos_comm.groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha']
    .diff().dt.days
)
valid_gaps = pedidos_comm[pedidos_comm['gap_dies'] > 0]

# ── Pedidos técnicos ──────────────────────────────────────────────────────────
tech_pedidos = (
    tech_base
    .groupby(['Id_Cliente', 'Num.Fact'], sort=False)
    .agg(data=('Fecha', 'min'), euros=('Valores_H', 'sum'))
    .reset_index()
    .sort_values(['Id_Cliente', 'data'])
)
tech_pedidos['gap_dies'] = (
    tech_pedidos.groupby('Id_Cliente')['data'].diff().dt.days
)

# ── Ciclos poblacionales (para clientes nuevos sin historial propio) ──────────
pop_comm = (
    valid_gaps.groupby('Familia_Potencial')['gap_dies']
    .agg(pop_cicle_mig='mean', pop_cicle_std='std')
    .reset_index()
)
bio_g = tech_pedidos[tech_pedidos['gap_dies'] > 0]['gap_dies']
pop_bio = pd.DataFrame([{
    'Familia_Potencial': 'Biomateriales',
    'pop_cicle_mig': bio_g.mean() if len(bio_g) > 0 else 60.0,
    'pop_cicle_std': bio_g.std()  if len(bio_g) > 1 else 20.0,
}])
pop_cycle = pd.concat([pop_comm, pop_bio], ignore_index=True)

print('✅ Preprocesamiento completado')
print(f'   Pares (cliente, familia): {monthly.groupby(["Id_Cliente","Familia_Potencial"]).ngroups:,}')
print(f'   Gaps válidos commodities: {len(valid_gaps):,}')
print()
print('Ciclos poblacionales:')
display(pop_cycle.round(1))


## 3. Construcción de la timeline unificada

In [ ]:
# Commodities: un pedido = una fila
comm_tl = (
    comm_base
    .groupby(['Id_Cliente', 'Familia_Potencial', 'Num.Fact'])
    .agg(Fecha=('Fecha', 'min'), euros=('Valores_H', 'sum'))
    .reset_index()
    [['Id_Cliente', 'Familia_Potencial', 'Fecha', 'euros']]
)

# Técnicos: un pedido = una fila
tech_tl = (
    tech_base
    .groupby(['Id_Cliente', 'Num.Fact'])
    .agg(Fecha=('Fecha', 'min'), euros=('Valores_H', 'sum'))
    .reset_index()
    .assign(Familia_Potencial='Biomateriales')
    [['Id_Cliente', 'Familia_Potencial', 'Fecha', 'euros']]
)

# Unión y normalización a día 0 = primera compra
timeline = (
    pd.concat([comm_tl, tech_tl], ignore_index=True)
    .sort_values(['Id_Cliente', 'Familia_Potencial', 'Fecha'])
    .reset_index(drop=True)
)
timeline['primer_pedido'] = (
    timeline.groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha'].transform('min')
)
timeline['dies_des_del_primer'] = (
    (timeline['Fecha'] - timeline['primer_pedido']).dt.days
)
timeline['n_pedidos_total'] = (
    timeline.groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha'].transform('count')
)

print(f'✅ Timeline: {len(timeline):,} pedidos  |  '
      f'{timeline.Id_Cliente.nunique():,} clientes  |  '
      f'{timeline.Familia_Potencial.nunique()} familias')


---

## 4. Explorador Interactivo con Alertas de Intervalo

### Leyenda del gráfico
- 🔵 **Barras azules** = cliente establecido (ciclo personal EWM)
- 🟠 **Barras naranjas** = cliente nuevo (ciclo poblacional de la familia)
- 🟢 **Zona verde** = ventana de compra esperada (ciclo ±0.5σ)
- 🔴 **Zona roja** = zona de riesgo (retraso >0.5σ)
- ⬛ **Línea negra vertical** = hoy simulado

### Nuevas alertas
| Alerta | Cuándo se activa | Qué hacer |
|--------|-----------------|-----------|
| ⚠️ **ANTICIPACIÓN** | Hoy ha entrado en la ventana de compra y el cliente NO ha comprado aún | Contactar ahora, antes de que compre a la competencia |
| 🔴 **CIERRE** | La ventana ha cerrado y el cliente NO compró en todo el intervalo | ¡Llamada urgente! Alta probabilidad de pedido a la competencia |


In [ ]:
FAMILIES = ['Anestesia', 'Bioseguridad', 'Biomateriales']

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_clients_for_familia(familia, min_n=1):
    mask = (timeline['Familia_Potencial'] == familia) & (timeline['n_pedidos_total'] >= min_n)
    return sorted(timeline.loc[mask, 'Id_Cliente'].unique().tolist())


def _get_client_timeline(client_id, familia):
    mask = (timeline['Id_Cliente'] == client_id) & (timeline['Familia_Potencial'] == familia)
    return timeline.loc[mask].sort_values('dies_des_del_primer').copy()


def _ewm_cycle(gaps_arr, half_life):
    n = len(gaps_arr)
    lam = np.log(2) / max(half_life, 0.1)
    weights = np.exp(lam * np.arange(n))
    weights /= weights.sum()
    cicle = float(np.dot(weights, gaps_arr))
    if n > 1:
        variance = float(np.dot(weights, (gaps_arr - cicle) ** 2))
        std = float(np.sqrt(variance))
    else:
        std = cicle * 0.30
    if np.isnan(std) or std <= 0:
        std = cicle * 0.30
    return cicle, std, weights


def _get_cycle(known_df, familia, es_nou, half_life):
    gaps = known_df['dies_des_del_primer'].diff().dropna()
    gaps = gaps[gaps > 0]
    if not es_nou and len(gaps) >= 1:
        cicle, cicle_std, weights = _ewm_cycle(gaps.values.astype(float), half_life)
        return cicle, cicle_std, weights, 'personal (EWM)', '#1565C0', '#1565C0'
    else:
        pop = pop_cycle[pop_cycle['Familia_Potencial'] == familia]
        cicle     = float(pop.iloc[0]['pop_cicle_mig']) if len(pop) > 0 else 45.0
        cicle_std = float(pop.iloc[0]['pop_cicle_std']) if len(pop) > 0 else 15.0
        if np.isnan(cicle_std) or cicle_std <= 0:
            cicle_std = cicle * 0.30
        return cicle, cicle_std, np.array([]), 'poblacional', '#E65100', '#E65100'


# ── Widgets ───────────────────────────────────────────────────────────────────

familia_w = widgets.Dropdown(
    options=FAMILIES, value='Anestesia',
    description='Familia:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='240px')
)
client_w = widgets.Dropdown(
    options=get_clients_for_familia('Anestesia'),
    description='Cliente ID:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)
min_pedidos_w = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='Min pedidos (establecido):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='440px')
)
dia_avui_w = widgets.IntSlider(
    value=1000, min=1, max=2000, step=5,
    description='Hoy simulado (días):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)
half_life_w = widgets.FloatSlider(
    value=3.0, min=0.5, max=10.0, step=0.5,
    description='Vida media (gaps):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='440px'),
    readout_format='.1f'
)
out = widgets.Output()


def _update_slider_for_client(client_id, familia):
    cdata = _get_client_timeline(client_id, familia)
    if len(cdata) == 0:
        return
    primer_date    = cdata['primer_pedido'].iloc[0]
    dies_avui_real = (REFERENCE_DATE - primer_date).days
    first_gap = int(cdata['dies_des_del_primer'].iloc[1]) if len(cdata) > 1 else 10
    dia_avui_w.min   = max(first_gap, 1)
    dia_avui_w.max   = dies_avui_real
    initial = max(int(dies_avui_real * 0.60), dia_avui_w.min)
    dia_avui_w.value = initial


def on_familia_change(change):
    new_clients = get_clients_for_familia(change['new'])
    client_w.options = new_clients
    if new_clients:
        client_w.value = new_clients[0]
        _update_slider_for_client(new_clients[0], change['new'])


def on_client_change(change):
    _update_slider_for_client(change['new'], familia_w.value)


familia_w.observe(on_familia_change, names='value')
client_w.observe(on_client_change,   names='value')


# ── Función principal de plot ─────────────────────────────────────────────────

def plot_client_timeline(client_id, familia, min_pedidos_establert, dia_avui_sim, half_life):
    cdata = _get_client_timeline(client_id, familia)
    if len(cdata) == 0:
        print(f'Cliente {client_id} sin datos para {familia}')
        return

    primer_date      = cdata['primer_pedido'].iloc[0]
    known  = cdata[cdata['dies_des_del_primer'] <= dia_avui_sim]
    future = cdata[cdata['dies_des_del_primer'] >  dia_avui_sim]
    n_known = len(known)
    if n_known == 0:
        print('Desplaza el slider a la derecha: no hay ninguna compra conocida.')
        return

    dies_ultim_known = int(known['dies_des_del_primer'].max())
    es_nou = n_known < min_pedidos_establert

    cicle, cicle_std, norm_w, cicle_tipus, bar_color, cicle_color = _get_cycle(
        known, familia, es_nou, half_life
    )
    gap_positions = known['dies_des_del_primer'].values[1:]

    # Ventanas de predicción
    preds = []
    for offset in range(1, 12):
        nd = dies_ultim_known + offset * cicle
        if nd > dies_ultim_known + 5 * cicle:
            break
        preds.append({
            'day':       nd,
            'low':       nd - 0.5 * cicle_std,
            'high':      nd + 0.5 * cicle_std,
            'risk_high': nd + 1.5 * cicle_std,
        })

    # ── DETECCIÓN DE ALERTAS ──────────────────────────────────────────────────
    anticipation_alerts = []
    cierre_alerts       = []

    for p in preds:
        purchases_in_window = known[
            (known['dies_des_del_primer'] >= p['low']) &
            (known['dies_des_del_primer'] <= p['high'])
        ]
        has_purchase = len(purchases_in_window) > 0

        # Alerta anticipación: hoy está DENTRO de la ventana pero sin compra
        if p['low'] <= dia_avui_sim <= p['high'] and not has_purchase:
            anticipation_alerts.append(p)

        # Alerta cierre: hoy está EN LA ZONA DE RIESGO (pasada la ventana) y sin compra
        elif p['high'] < dia_avui_sim <= p['risk_high'] and not has_purchase:
            cierre_alerts.append(p)

    # ── Backtesting hit/miss ──────────────────────────────────────────────────
    hit_set = set()
    for _, row in future.iterrows():
        d = row['dies_des_del_primer']
        if any(p['low'] <= d <= p['high'] for p in preds):
            hit_set.add(d)

    n_future    = len(future)
    n_hits      = len(hit_set)
    hit_rate    = n_hits / n_future if n_future > 0 else None
    future_hit  = future[future['dies_des_del_primer'].isin(hit_set)]
    future_miss = future[~future['dies_des_del_primer'].isin(hit_set)]

    # ── PLOT ──────────────────────────────────────────────────────────────────
    fig, axes2 = plt.subplots(
        2, 1, figsize=(15, 6.5),
        gridspec_kw={'height_ratios': [5, 1.2], 'hspace': 0.06}
    )
    ax, ax_w = axes2
    ymax  = float(cdata['euros'].max()) if cdata['euros'].max() > 0 else 1.0
    bar_w = max(cicle * 0.05, 3)

    # Ventanas de predicción
    for p in preds:
        ax.axvspan(p['low'],  p['high'],      alpha=0.14, color='green', zorder=1)
        ax.axvspan(p['high'], p['risk_high'], alpha=0.08, color='red',   zorder=1)
        ax.axvline(p['day'], color=cicle_color, ls='--', lw=0.9, alpha=0.5, zorder=2)

    # ── Marcadores de alerta ──────────────────────────────────────────────────
    for i, p in enumerate(anticipation_alerts):
        ax.axvline(p['low'], color='#FF6F00', lw=3, zorder=8, alpha=0.9)
        y_ann = ymax * (0.82 - i * 0.12)
        ax.annotate(
            '\u26a0\ufe0f INICIO VENTANA\nCONTACTAR AHORA',
            xy=(p['low'], y_ann * 0.88),
            xytext=(p['low'] + cicle * 0.15, y_ann),
            fontsize=7.5, color='#BF360C', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#FF6F00', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0',
                      edgecolor='#FF6F00', lw=1.5),
            zorder=9
        )

    for i, p in enumerate(cierre_alerts):
        ax.axvline(p['high'], color='#B71C1C', lw=3, zorder=8, alpha=0.9)
        y_ann = ymax * (0.65 - i * 0.12)
        ax.annotate(
            '\U0001f534 FIN VENTANA\nSIN COMPRA — \u00a1URGENTE!',
            xy=(p['high'], y_ann * 0.88),
            xytext=(p['high'] + cicle * 0.15, y_ann),
            fontsize=7.5, color='#B71C1C', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#B71C1C', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFEBEE',
                      edgecolor='#B71C1C', lw=1.5),
            zorder=9
        )

    # Barras historial con intensidad EWM
    known_rows = list(known.iterrows())
    for idx, (_, row) in enumerate(known_rows):
        d = row['dies_des_del_primer']
        if idx > 0 and len(norm_w) > 0 and (idx - 1) < len(norm_w):
            w_min, w_max = norm_w.min(), norm_w.max()
            alpha_bar = 0.28 + 0.67 * (norm_w[idx - 1] - w_min) / max(w_max - w_min, 1e-9)
        else:
            alpha_bar = 0.40
        ax.bar(d, row['euros'], width=bar_w, color=bar_color, alpha=alpha_bar, zorder=3)
        if row['euros'] > 0:
            ax.text(d, row['euros'] + ymax * 0.015, f'{row["euros"]:.0f}',
                    ha='center', va='bottom', fontsize=7.5, color='#222')

    if len(future_hit) > 0:
        ax.bar(future_hit['dies_des_del_primer'], future_hit['euros'],
               width=bar_w, color='#2E7D32', alpha=0.75, zorder=4)
    if len(future_miss) > 0:
        ax.bar(future_miss['dies_des_del_primer'], future_miss['euros'],
               width=bar_w, color='#90A4AE', alpha=0.55, zorder=3)
    for _, row in future.iterrows():
        if row['euros'] > 0:
            c = '#1B5E20' if row['dies_des_del_primer'] in hit_set else '#546E7A'
            ax.text(row['dies_des_del_primer'], row['euros'] + ymax * 0.015,
                    f'{row["euros"]:.0f}', ha='center', va='bottom',
                    fontsize=7.5, color=c)

    ax.axvline(dia_avui_sim, color='black', lw=2.4, zorder=6)

    estat   = 'NUEVO' if es_nou else 'ESTABLECIDO'
    hit_txt = (f'  Tasa acierto: {n_hits}/{n_future} = {hit_rate*100:.0f}%'
               if hit_rate is not None else '  Sin compras futuras')
    ax.set_title(
        f'Cliente {client_id}  |  {familia}  |  {estat}  |  '
        f'Ciclo {cicle_tipus}: {cicle:.0f} días (+/-{cicle_std:.0f}){hit_txt}\n'
        f'Intensidad barras = peso EWM  |  half_life={half_life:.1f} gaps',
        fontsize=10, pad=8
    )
    ax.set_ylabel('Importe (€)', fontsize=10)

    handles = [
        mpatches.Patch(color=bar_color, alpha=0.75,
                       label=f'Historial ({n_known} pedidos) — intensidad = peso EWM'),
        mpatches.Patch(color='green', alpha=0.30, label='Ventana esperada (±0.5σ)'),
        mpatches.Patch(color='red',   alpha=0.20, label='Zona de riesgo (>0.5σ retraso)'),
        plt.Line2D([0], [0], color='black', lw=2,   label=f'Hoy simulado (día {dia_avui_sim})'),
        plt.Line2D([0], [0], color='#FF6F00', lw=2.5, label='⚠️ Alerta Anticipación (inicio ventana)'),
        plt.Line2D([0], [0], color='#B71C1C', lw=2.5, label='🔴 Alerta Cierre (fin ventana sin compra)'),
    ]
    if len(future_hit)  > 0:
        handles.append(mpatches.Patch(color='#2E7D32', alpha=0.75, label=f'Acierto ({len(future_hit)})'))
    if len(future_miss) > 0:
        handles.append(mpatches.Patch(color='#90A4AE', alpha=0.55, label=f'No acierto ({len(future_miss)})'))
    ax.legend(handles=handles, loc='upper left', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

    x_right = max(dia_avui_sim * 1.05,
                  dies_ultim_known + 3.5 * cicle,
                  cdata['dies_des_del_primer'].max() * 1.03)
    ax.set_xlim(-bar_w * 2, x_right)
    ax.set_ylim(0, ymax * 1.22)
    ax.set_xticklabels([])

    # Sub-plot pesos EWM
    if len(norm_w) > 0:
        cmap_vals   = 0.3 + 0.7 * norm_w / norm_w.max()
        bar_colors_w = plt.cm.Blues(cmap_vals)
        ax_w.bar(gap_positions, norm_w * 100, width=bar_w * 1.5,
                 color=bar_colors_w, alpha=0.9)
        for gp, gw in zip(gap_positions, norm_w):
            ax_w.text(gp, gw * 100 + norm_w.max() * 2, f'{gw*100:.1f}%',
                      ha='center', va='bottom', fontsize=7, color='#333')
        ax_w.axvline(dia_avui_sim, color='black', lw=1.5, alpha=0.4)
        ax_w.set_xlim(ax.get_xlim())
        ax_w.set_ylim(0, norm_w.max() * 140)
        ax_w.set_ylabel('Peso EWM\n(%)', fontsize=7.5)
        ax_w.grid(axis='y', alpha=0.2)
        ax_w.tick_params(labelsize=7)
    else:
        ax_w.axis('off')
    ax_w.set_xlabel('Días desde el primer pedido  (día 0 = primera compra)', fontsize=10)

    plt.tight_layout()
    plt.show()

    # ── Resumen texto ─────────────────────────────────────────────────────────
    print(f"{'─'*62}")
    print(f"  Backtesting EWM  |  Cliente {client_id}  |  {familia}")
    print(f"{'─'*62}")
    print(f"  Hoy simulado:    día {dia_avui_sim}  "
          f"({(primer_date + pd.Timedelta(days=dia_avui_sim)).strftime('%d/%m/%Y')})")
    print(f"  Historial:       {n_known} pedidos  (hasta día {dies_ultim_known})")
    if len(norm_w) > 0:
        print(f"  EWM half_life:   {half_life:.1f} gaps  "
              f"→ peso último gap: {norm_w[-1]*100:.1f}%  |  primer gap: {norm_w[0]*100:.1f}%")
    print(f"  Ciclo EWM:       {cicle:.0f} días ± {cicle_std:.0f}  ({cicle_tipus})")
    print(f"  Compras futuras: {n_future}")
    if hit_rate is not None:
        stars = '★' * min(int(hit_rate * 5), 5)
        print(f"  Aciertos:        {n_hits} / {n_future}  ({hit_rate*100:.0f}%)  {stars}")
    print(f"{'─'*62}")

    # ── Bloque de alertas ─────────────────────────────────────────────────────
    print()
    if anticipation_alerts or cierre_alerts:
        print(f"{'═'*62}")
        print(f"  🚨 ALERTAS ACTIVAS EN EL MOMENTO SIMULADO")
        print(f"{'═'*62}")
        for p in anticipation_alerts:
            d_ini  = (primer_date + pd.Timedelta(days=int(p['low']))).strftime('%d/%m/%Y')
            d_fin  = (primer_date + pd.Timedelta(days=int(p['high']))).strftime('%d/%m/%Y')
            print(f"  ⚠️  ALERTA ANTICIPACIÓN — INICIO DE VENTANA DE COMPRA")
            print(f"     El cliente HA ENTRADO en su ventana prevista ({d_ini} → {d_fin}).")
            print(f"     No se ha registrado ninguna compra todavía.")
            print(f"     ACCIÓN: Contactar AHORA para asegurar el pedido antes que la")
            print(f"             competencia. Es el momento óptimo de intervención.")
            print()
        for p in cierre_alerts:
            d_ini  = (primer_date + pd.Timedelta(days=int(p['low']))).strftime('%d/%m/%Y')
            d_fin  = (primer_date + pd.Timedelta(days=int(p['high']))).strftime('%d/%m/%Y')
            d_risk = (primer_date + pd.Timedelta(days=int(p['risk_high']))).strftime('%d/%m/%Y')
            print(f"  🔴 ALERTA CIERRE — FIN DE VENTANA SIN COMPRA")
            print(f"     La ventana prevista ({d_ini} → {d_fin}) ha CERRADO sin compra.")
            print(f"     Zona de riesgo activa hasta: {d_risk}")
            print(f"     ACCIÓN: ¡Llamada URGENTE! Alta probabilidad de que el cliente")
            print(f"             haya comprado (o esté a punto de comprar) a la competencia.")
            print()
        print(f"{'═'*62}")
    else:
        print(f"  ✅ Sin alertas de intervalo activas en el momento simulado.")
        print(f"{'═'*62}")


def update_chart(change=None):
    with out:
        clear_output(wait=True)
        if client_w.value is not None:
            plot_client_timeline(
                client_w.value, familia_w.value,
                min_pedidos_w.value, dia_avui_w.value,
                half_life_w.value
            )


familia_w.observe(update_chart,     names='value')
client_w.observe(update_chart,      names='value')
min_pedidos_w.observe(update_chart, names='value')
dia_avui_w.observe(update_chart,    names='value')
half_life_w.observe(update_chart,   names='value')

header = widgets.HTML(
    '<h3 style="margin:4px 0;color:#1565C0">Explorador + Alertas de Intervalo (EWM)</h3>'
    '<p style="color:#555;margin:2px 0">'
    'Ciclo con <b>decaimiento exponencial</b>: los gaps más recientes pesan más. '
    '<b>⚠️ Naranja</b> = alerta al inicio del intervalo de compra (anticipación). '
    '<b>🔴 Rojo</b> = alerta al final del intervalo si el cliente no ha comprado.</p>'
)
row1 = widgets.HBox([familia_w, client_w],
                    layout=widgets.Layout(gap='10px', align_items='center'))
row2 = widgets.HBox([min_pedidos_w, half_life_w],
                    layout=widgets.Layout(gap='10px', align_items='center'))
row3 = widgets.HBox([dia_avui_w])
display(widgets.VBox([header, row1, row2, row3, out]))

_update_slider_for_client(client_w.value, familia_w.value)
update_chart()


---

## 5. Share of Wallet y Velocidad del Share (Clientes Promiscuos)

### 1. ¿Cuál es la métrica clave? (El "Share of Wallet")
Para un cliente promiscuo, no solo nos importa cuánto nos compra en euros, sino *qué porcentaje de su capacidad total de compra (potencial) nos está destinando a nosotros*. A esto se le llama **Share of Wallet**.

- **Cálculo:** Se suman las ventas de los últimos 12 meses (`rolling_12m`) y se dividen entre el potencial anual del cliente (`Potencial_EUR_anual`).

### 2. ¿Cómo se detecta el momento óptimo? (La "Velocidad del Share")
El modelo no se fija solo en la "foto actual" (qué porcentaje de share tenemos hoy), sino en la *película* (hacia dónde va la tendencia). Para ello se calcula la **Velocidad del Share** (técnicamente, la primera derivada), que es simplemente la diferencia del Share entre este mes y el mes anterior.

Esto genera dos escenarios claros de actuación:

- **Velocidad Positiva (Barras Verdes en el gráfico):** El cliente está acelerando sus compras con nosotros y quitándoselas a la competencia.
  - **Acción de Negocio:** Es el momento de **fidelizar e intentar ventas cruzadas (cross-sell)**, ya que el cliente está receptivo a nuestra marca.

- **Velocidad Negativa (Barras Rojas en el gráfico):** Nuestro Share está cayendo. El cliente está empezando a desviar sus compras hacia un competidor.
  - **Acción de Negocio (Riesgo Inminente):** Si la velocidad de caída es drástica (umbral: caída > 5% mensual), salta una alerta. El comercial debe llamar inmediatamente, posiblemente con una **oferta agresiva** para frenar la fuga antes de que el competidor se haga con todo el negocio.

### En resumen para el equipo de ventas
El modelo de "clientes promiscuos" les dice a los comerciales que dejen de mirar solo la facturación absoluta. Les avisa de forma temprana (basándose en los cambios de tendencia o *momentum*) cuándo un cliente está a punto de irse con la competencia para que actúen con urgencia, o cuándo está en un ciclo de "enamoramiento" con Inibsa para que aprovechen y le vendan más cosas.

| Velocidad | Señal | Acción |
|-----------|-------|--------|
| > +5pp/mes (🟢) | Cliente acelerando con nosotros | Cross-sell, fidelizar |
| Entre -5 y +5pp/mes (🔵) | Share estable | Seguimiento estándar |
| < -5pp/mes (🔴) | Fuga hacia competidor | Llamada urgente con oferta agresiva |


In [ ]:
# ── Widgets Share of Wallet ───────────────────────────────────────────────────

SOW_FAMILIES = ['Anestesia', 'Bioseguridad']  # Solo familias con datos de potencial

def get_sow_clients(familia, min_meses=3):
    mask = (
        (monthly['Familia_Potencial'] == familia) &
        (monthly['potencial_eur'] > 0)
    )
    counts = monthly[mask].groupby('Id_Cliente')['year_month'].count()
    return sorted(counts[counts >= min_meses].index.tolist())


sow_familia_w = widgets.Dropdown(
    options=SOW_FAMILIES, value='Anestesia',
    description='Familia:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='240px')
)
sow_client_w = widgets.Dropdown(
    options=get_sow_clients('Anestesia'),
    description='Cliente ID:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)
out_sow = widgets.Output()


def on_sow_familia_change(change):
    new_clients = get_sow_clients(change['new'])
    sow_client_w.options = new_clients
    if new_clients:
        sow_client_w.value = new_clients[0]


sow_familia_w.observe(on_sow_familia_change, names='value')


def plot_share_wallet(client_id, familia):
    mask = (monthly['Id_Cliente'] == client_id) & (monthly['Familia_Potencial'] == familia)
    cm   = monthly[mask].copy().sort_values('year_month_dt')

    if len(cm) < 3:
        print("Datos insuficientes (mínimo 3 meses de historial).")
        return

    potencial = cm['potencial_eur'].iloc[0]
    if potencial <= 0:
        print("Potencial del cliente = 0. No se puede calcular el Share of Wallet.")
        return

    # ── Reindexar a TODOS los meses del calendario ────────────────────────────
    # Sin este paso los meses sin compra no tienen fila: el rolling(12) los
    # ignora y las ventas de hace >12 meses nunca salen de la ventana,
    # dejando el share artificialmente pegado en 100% aunque el cliente
    # lleve años sin comprar.
    cm = cm.set_index('year_month_dt')
    all_months = pd.date_range(cm.index.min(), REFERENCE_DATE, freq='MS')
    cm = cm.reindex(all_months)
    cm['euros_venuts']  = cm['euros_venuts'].fillna(0)
    cm['potencial_eur'] = cm['potencial_eur'].ffill().bfill()

    # Cálculo de Share y Velocidad sobre la serie completa de meses
    cm['rolling_12m']    = cm['euros_venuts'].rolling(12, min_periods=1).sum()
    cm['share']          = (cm['rolling_12m'] / potencial).clip(0, 1)
    cm['share_velocity'] = cm['share'].diff()
    cm = cm.reset_index().rename(columns={'index': 'year_month_dt'})

    # ── Gráfico ────────────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(13, 7),
        gridspec_kw={'height_ratios': [2.5, 1.5], 'hspace': 0.10}
    )

    # Top: Share of Wallet
    ax1.plot(cm['year_month_dt'], cm['share'], color='#1565C0',
             marker='o', lw=2.2, markersize=3, label='Share of Wallet (rolling 12m)')
    ax1.fill_between(cm['year_month_dt'], cm['share'], alpha=0.08, color='#1565C0')
    ax1.axhline(0.70, color='#2E7D32', ls='--', lw=1.2, alpha=0.7, label='70% (umbral fidelidad)')
    ax1.axhline(0.20, color='#C62828', ls='--', lw=1.2, alpha=0.7, label='20% (umbral riesgo)')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax1.set_ylabel('Share of Wallet', fontsize=10)
    ax1.set_ylim(0, 1.1)
    ax1.set_title(
        f'Share of Wallet — Cliente {client_id} | {familia}\n'
        f'Potencial anual: {potencial:,.0f} €  |  '
        f'Capturado últimos 12m: {cm["rolling_12m"].iloc[-1]:,.0f} €  |  '
        f'Recuperable: {max(0, potencial - cm["rolling_12m"].iloc[-1]):,.0f} €',
        fontsize=11
    )
    ax1.legend(fontsize=9)
    ax1.set_xticklabels([])
    ax1.grid(axis='y', alpha=0.3)

    # Bottom: Velocidad del Share
    vel_pp = cm['share_velocity'].fillna(0) * 100  # en puntos porcentuales
    colors_vel = ['#2E7D32' if v >= 0 else '#C62828' for v in vel_pp]
    ax2.bar(cm['year_month_dt'], vel_pp, color=colors_vel, alpha=0.75, width=25)
    ax2.axhline(0,   color='gray',    lw=1)
    ax2.axhline(-5,  color='#C62828', lw=1.3, ls='--', label='Umbral riesgo (-5pp/mes)')
    ax2.axhline(5,   color='#2E7D32', lw=1.3, ls='--', label='Umbral oportunidad (+5pp/mes)')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.1f}pp'))
    ax2.set_ylabel('Velocidad del Share\n(pp/mes)', fontsize=10)
    ax2.set_xlabel('Fecha', fontsize=10)
    ax2.legend(fontsize=8)
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ── Resumen y alerta ──────────────────────────────────────────────────────
    last_vel    = float(cm['share_velocity'].iloc[-1]) if len(cm) > 1 else 0.0
    last_share  = float(cm['share'].iloc[-1])
    last_12m    = float(cm['rolling_12m'].iloc[-1])
    recuperable = max(0.0, potencial - last_12m)

    print(f"{'═'*62}")
    print(f"  Análisis Share of Wallet — Cliente {client_id} | {familia}")
    print(f"{'═'*62}")
    print(f"  Share actual (rolling 12m): {last_share:.1%}")
    print(f"  Velocidad actual:           {last_vel:+.1%}/mes")
    print(f"  Potencial anual:            {potencial:,.0f} €")
    print(f"  Euros capturados (12m):     {last_12m:,.0f} €")
    print(f"  Euros recuperables:         {recuperable:,.0f} €")
    print(f"{'─'*62}")

    if last_vel < -0.05:
        print(f"  🔴 ALERTA: RIESGO INMINENTE DE FUGA")
        print(f"     El Share está cayendo {last_vel:.1%} por mes.")
        print(f"     El cliente está desviando compras hacia la competencia.")
        print(f"     ACCIÓN: Llamar con OFERTA AGRESIVA para frenar la fuga.")
        print(f"             Potencial en riesgo: {recuperable:,.0f} € adicionales.")
    elif last_vel > 0.05:
        print(f"  🟢 OPORTUNIDAD: CLIENTE EN FASE DE FIDELIZACIÓN")
        print(f"     El Share está creciendo {last_vel:.1%} por mes.")
        print(f"     El cliente está receptivo a nuestra marca.")
        print(f"     ACCIÓN: Intentar CROSS-SELL — el cliente está en su mejor")
        print(f"             momento de receptividad. Recuperable: {recuperable:,.0f} €")
    elif last_share < 0.20:
        print(f"  🟡 ATENCIÓN: Share muy bajo ({last_share:.1%})")
        print(f"     La competencia domina a este cliente.")
        print(f"     ACCIÓN: Revisar estrategia de cuenta. Oportunidad de {recuperable:,.0f} €")
    else:
        print(f"  🔵 SEGUIMIENTO ESTÁNDAR")
        print(f"     Share estable ({last_share:.1%}). Mantener contacto habitual.")
    print(f"{'═'*62}")


def update_sow(change=None):
    with out_sow:
        clear_output(wait=True)
        if sow_client_w.value is not None:
            plot_share_wallet(sow_client_w.value, sow_familia_w.value)


sow_familia_w.observe(update_sow, names='value')
sow_client_w.observe(update_sow,  names='value')

sow_header = widgets.HTML(
    '<h3 style="margin:4px 0;color:#E65100">Share of Wallet — Explorador por Cliente</h3>'
    '<p style="color:#555;margin:2px 0">'
    '<b>🟢 Verde</b> = velocidad positiva (ganar terreno). '
    '<b>🔴 Rojo</b> = velocidad negativa (perder terreno). '
    'Umbral de alerta: ±5 puntos porcentuales por mes.</p>'
)
sow_row1 = widgets.HBox([sow_familia_w, sow_client_w],
                         layout=widgets.Layout(gap='10px', align_items='center'))
display(widgets.VBox([sow_header, sow_row1, out_sow]))

update_sow()


---

## 6. Alerta de Fuga — Detección de Deterioro de Tendencia

> **Modo backtesting**: elige cualquier mes del historial como "hoy simulado" y el sistema detecta qué alerta habría disparado en ese momento.

### Lógica del modelo

El sistema extrae la **tendencia subyacente** de las compras mensuales hasta el "hoy simulado" y detecta si esa tendencia ha cambiado de dirección en los últimos N meses.

**Paso 1 — Extracción de tendencia**
- ≥ 18 meses: **STL** (Seasonal-Trend decomposition), extrae la tendencia pura eliminando estacionalidad y ruido.
- < 18 meses: **EWM** (media exponencial ponderada), suaviza dando más peso a los datos recientes.

**Paso 2 — Detección del cambio (regresión dual)**

Dos regresiones sobre la tendencia extraída hasta "hoy":
- 🟢 **Recta histórica** (todo lo anterior a la ventana reciente): pendiente de referencia del cliente.
- 🔴 **Recta reciente** (últimos N meses hasta hoy): pendiente actual.

**Niveles de alerta**

| Nivel | Condición | Acción |
|-------|-----------|--------|
| 🔴 **FUGA INMINENTE** | Pendiente reciente negativa + significativa + peor que histórico | Llamada urgente con oferta agresiva |
| 🟠 **DETERIORO** | Pendiente reciente negativa + significativa | Revisar relación comercial |
| 🟡 **VIGILAR** | Pendiente negativa pero no significativa | Monitorizar próximos meses |
| 🟢 **OK** | Tendencia estable o creciente | Seguimiento estándar |


In [ ]:
from scipy import stats as sp_stats

try:
    from statsmodels.tsa.seasonal import STL
    HAS_STL = True
except ImportError:
    HAS_STL = False
    print("⚠️  statsmodels no disponible — se usará EWM.")

# ── Helpers ───────────────────────────────────────────────────────────────────

def _extract_trend(series_indexed):
    n = len(series_indexed)
    if HAS_STL and n >= 18:
        try:
            stl = STL(series_indexed, period=12, robust=True, seasonal=7).fit()
            tag = 'STL·LOESS' if n >= 24 else 'STL·LOESS ⚠️ borde'
            return stl.trend.values, tag
        except Exception:
            pass
    span = max(3, n // 4)
    return series_indexed.ewm(span=span, adjust=False).mean().values, f'EWM (span={span}m)'


def _classify_alert(slope_rec, slope_hist, p_rec):
    is_neg   = slope_rec < 0
    is_sig   = p_rec < 0.15
    is_worse = slope_rec < min(0.0, slope_hist)
    if is_neg and is_sig and is_worse:  return 'FUGA_INMINENTE'
    if is_neg and is_sig:               return 'DETERIORO'
    if is_neg:                          return 'VIGILAR'
    return 'OK'


# ── Estado compartido ─────────────────────────────────────────────────────────
_fuga_all_months = pd.DatetimeIndex([])

def _rebuild_fuga_state(client_id, familia):
    global _fuga_all_months
    mask = (monthly['Id_Cliente'] == client_id) & (monthly['Familia_Potencial'] == familia)
    cm   = monthly[mask]
    if len(cm) == 0:
        _fuga_all_months = pd.DatetimeIndex([])
        return
    _fuga_all_months = pd.date_range(cm['year_month_dt'].min(), REFERENCE_DATE, freq='MS')


# ── Widgets ───────────────────────────────────────────────────────────────────
FUGA_FAMILIES = ['Anestesia', 'Bioseguridad']

def get_fuga_clients(familia, min_meses=6):
    mask = (monthly['Familia_Potencial'] == familia) & (monthly['potencial_eur'] > 0)
    counts = monthly[mask].groupby('Id_Cliente')['year_month'].count()
    return sorted(counts[counts >= min_meses].index.tolist())


fuga_familia_w = widgets.Dropdown(
    options=FUGA_FAMILIES, value='Anestesia',
    description='Familia:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='240px')
)
fuga_client_w = widgets.Dropdown(
    options=get_fuga_clients('Anestesia'),
    description='Cliente ID:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)

# Slider 1 — "Hoy simulado": mes del historial que tratamos como presente
fuga_dia_avui_w = widgets.IntSlider(
    value=0, min=0, max=60, step=1,
    description='Hoy simulado (mes #):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='640px')
)

# Slider 2 — Ventana de análisis: cuántos meses ANTES del "hoy simulado" examinar
fuga_n_recent_w = widgets.IntSlider(
    value=12, min=3, max=36, step=1,
    description='Ventana análisis (últimos N meses):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='460px')
)

fuga_date_label = widgets.HTML(value='')
out_fuga = widgets.Output()


def _update_fuga_label():
    idx      = fuga_dia_avui_w.value
    n_recent = fuga_n_recent_w.value
    months   = _fuga_all_months
    n        = len(months)
    if n == 0 or idx >= n:
        fuga_date_label.value = ''
        return
    win_start = max(0, idx - n_recent + 1)
    n_hist    = win_start
    d_hoy     = months[idx].strftime('%b %Y')
    d_ini     = months[0].strftime('%b %Y')
    d_hist_end= months[max(0, win_start - 1)].strftime('%b %Y') if win_start > 0 else '—'
    d_rec_ini = months[win_start].strftime('%b %Y')
    fuga_date_label.value = (
        f'<span style="font-size:12px;color:#444">'
        f'📅 Hoy simulado: <b>{d_hoy}</b>&nbsp;&nbsp;|&nbsp;&nbsp;'
        f'📋 Histórico: <b>{d_ini} → {d_hist_end}</b> ({n_hist}m)&nbsp;&nbsp;'
        f'🔍 Análisis: <b>{d_rec_ini} → {d_hoy}</b> ({n_recent}m)</span>'
    )

fuga_dia_avui_w.observe(lambda c: _update_fuga_label(), names='value')
fuga_n_recent_w.observe(lambda c: _update_fuga_label(), names='value')


def _update_fuga_sliders(client_id, familia):
    _rebuild_fuga_state(client_id, familia)
    n = len(_fuga_all_months)
    if n < 4:
        return
    fuga_dia_avui_w.max   = n - 1
    fuga_dia_avui_w.value = n - 1          # por defecto: el último mes = hoy real
    fuga_n_recent_w.max   = max(3, n - 3)
    fuga_n_recent_w.value = min(12, max(3, n - 3))
    _update_fuga_label()


def on_fuga_familia_change(change):
    nc = get_fuga_clients(change['new'])
    fuga_client_w.options = nc
    if nc:
        fuga_client_w.value = nc[0]
        _update_fuga_sliders(nc[0], change['new'])

def on_fuga_client_change(change):
    _update_fuga_sliders(change['new'], fuga_familia_w.value)

fuga_familia_w.observe(on_fuga_familia_change, names='value')
fuga_client_w.observe(on_fuga_client_change,   names='value')


# ── Función principal ─────────────────────────────────────────────────────────

def plot_fuga_alert(client_id, familia, dia_avui_idx, n_recent):
    mask = (monthly['Id_Cliente'] == client_id) & (monthly['Familia_Potencial'] == familia)
    cm   = monthly[mask].copy().sort_values('year_month_dt')
    if len(cm) == 0:
        print(f"Cliente {client_id} sin datos para {familia}.")
        return

    # Reindexar al calendario completo hasta REFERENCE_DATE
    cm = cm.set_index('year_month_dt')
    all_months = pd.date_range(cm.index.min(), REFERENCE_DATE, freq='MS')
    cm = cm.reindex(all_months)
    cm['euros_venuts']  = cm['euros_venuts'].fillna(0)
    cm['potencial_eur'] = cm['potencial_eur'].ffill().bfill()
    cm = cm.reset_index().rename(columns={'index': 'year_month_dt'})
    n_total = len(cm)

    if n_total < 4:
        print("Datos insuficientes.")
        return

    # "Hoy simulado" = mes en el índice dia_avui_idx
    dia_avui_idx = min(dia_avui_idx, n_total - 1)
    today_date   = cm['year_month_dt'].iloc[dia_avui_idx]

    # Solo usamos datos HASTA el hoy simulado (no miramos el futuro)
    cm_known = cm.iloc[:dia_avui_idx + 1].copy()
    n = len(cm_known)

    if n < 4:
        print("Mueve el slider de 'Hoy simulado' más a la derecha para tener más datos.")
        return

    # Ventana de análisis: últimos n_recent meses hasta hoy simulado
    n_recent  = min(n_recent, n - 2)
    win_start = n - n_recent
    n_hist    = win_start

    if n_recent < 2:
        print("Ventana demasiado corta — mueve 'Ventana análisis' a la derecha.")
        return

    # ── Extracción de tendencia hasta el hoy simulado ─────────────────────────
    series = cm_known.set_index('year_month_dt')['euros_venuts']
    trend_vals, method = _extract_trend(series)

    # ── Regresión histórica (antes de la ventana reciente) ────────────────────
    if n_hist >= 2:
        x_h = np.arange(n_hist, dtype=float)
        slope_hist, icept_hist, _, _, _ = sp_stats.linregress(x_h, trend_vals[:n_hist])
        y_hist_full = slope_hist * np.arange(n, dtype=float) + icept_hist
    else:
        slope_hist  = 0.0
        y_hist_full = np.full(n, trend_vals.mean())

    # ── Regresión ventana reciente ────────────────────────────────────────────
    x_r = np.arange(n_recent, dtype=float)
    slope_rec, icept_rec, _, p_rec, _ = sp_stats.linregress(x_r, trend_vals[win_start:])
    y_rec = slope_rec * x_r + icept_rec

    # ── Clasificación ─────────────────────────────────────────────────────────
    alert = _classify_alert(slope_rec, slope_hist, p_rec)

    mean_sales = cm_known['euros_venuts'][cm_known['euros_venuts'] > 0].mean() or 1.0
    rel_hist   = slope_hist / mean_sales
    rel_rec    = slope_rec  / mean_sales
    level_now  = float(trend_vals[-1])

    ALERT_COLOR = {'FUGA_INMINENTE': '#B71C1C', 'DETERIORO': '#E65100',
                   'VIGILAR': '#F9A825', 'OK': '#2E7D32'}
    ALERT_ICON  = {'FUGA_INMINENTE': '🔴', 'DETERIORO': '🟠', 'VIGILAR': '🟡', 'OK': '🟢'}
    rec_color  = '#B71C1C' if slope_rec < 0 else '#2E7D32'
    split_date = cm_known['year_month_dt'].iloc[win_start]

    # ── PLOT ──────────────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(14, 8),
        gridspec_kw={'height_ratios': [3, 2], 'hspace': 0.10}
    )
    x_min = cm_known['year_month_dt'].iloc[0] - pd.Timedelta(days=15)
    x_max = today_date + pd.Timedelta(days=45)

    for ax in (ax1, ax2):
        ax.set_xlim(x_min, x_max)

    # ── Subplot superior ──────────────────────────────────────────────────────
    ax1.bar(cm_known['year_month_dt'], cm_known['euros_venuts'],
            width=25, alpha=0.28, color='#1565C0', label='Ventas mensuales (reales)')
    ax1.plot(cm_known['year_month_dt'], trend_vals,
             color='#1565C0', lw=2.5, label=f'Tendencia ({method})')

    if n_hist >= 2:
        ax1.plot(cm_known['year_month_dt'], y_hist_full,
                 color='#2E7D32', lw=1.4, ls='--', alpha=0.75,
                 label=f'Ref. histórica  {slope_hist:+.1f} €/mes  ({rel_hist:+.1%}/mes)')

    ax1.plot(cm_known['year_month_dt'].iloc[win_start:], y_rec,
             color=rec_color, lw=2.8,
             label=(f'Tendencia reciente (últimos {n_recent}m)  {slope_rec:+.1f} €/mes  '
                    f'({rel_rec:+.1%}/mes, p={p_rec:.2f})'))

    # Línea vertical "hoy"
    ax1.axvline(today_date, color='black', lw=2.2, ls='--', alpha=0.7, zorder=6)

    ax1.axvline(split_date, color='gray', lw=1.2, ls=':', alpha=0.8)
    ymax_ax = (cm_known['euros_venuts'].max() or 1.0) * 1.05
    ax1.text(split_date - pd.Timedelta(days=5), ymax_ax * 0.97,
             f'← histórico ({n_hist}m)', fontsize=8, color='#555',
             va='top', ha='right')
    ax1.text(split_date + pd.Timedelta(days=5), ymax_ax * 0.97,
             f'análisis ({n_recent}m) →', fontsize=8, color=rec_color,
             va='top', ha='left')
    ax1.text(today_date + pd.Timedelta(days=6), ymax_ax * 0.92,
             f'HOY\n{today_date.strftime("%b %Y")}', fontsize=8.5,
             color='black', va='top', ha='left', fontweight='bold')

    ax1.set_ylabel('Ventas mensuales (€)', fontsize=10)
    ax1.set_title(
        f'Análisis de Tendencia — Cliente {client_id} | {familia}   '
        f'[{ALERT_ICON[alert]}  {alert.replace("_", " ")}]',
        fontsize=11, color=ALERT_COLOR[alert]
    )
    ax1.legend(fontsize=8, loc='upper left')
    ax1.set_xticklabels([])
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim(bottom=0)

    # ── Subplot inferior ──────────────────────────────────────────────────────
    ax2.plot(cm_known['year_month_dt'], trend_vals, color='#1565C0', lw=2.5, label='Tendencia')

    if n_hist >= 2:
        ax2.plot(cm_known['year_month_dt'].iloc[:win_start], y_hist_full[:n_hist],
                 color='#2E7D32', lw=1.4, ls='--', alpha=0.75,
                 label=f'Fit histórico  {slope_hist:+.1f} €/mes')

    ax2.fill_between(cm_known['year_month_dt'].iloc[win_start:], trend_vals[win_start:],
                     color='#FFCDD2' if slope_rec < 0 else '#C8E6C9',
                     alpha=0.40, label=f'Ventana análisis ({n_recent}m)')
    ax2.plot(cm_known['year_month_dt'].iloc[win_start:], y_rec,
             color=rec_color, lw=2.2)

    ax2.axvline(today_date, color='black', lw=2.2, ls='--', alpha=0.7, zorder=6)
    ax2.axvline(split_date, color='gray', lw=1.2, ls=':', alpha=0.8)
    ax2.set_ylabel('Tendencia (€)', fontsize=10)
    ax2.set_xlabel(f'Fecha  (hoy simulado = {today_date.strftime("%b %Y")})', fontsize=10)
    ax2.legend(fontsize=8, loc='upper left')
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim(bottom=0)

    plt.tight_layout()
    plt.show()

    # ── Resumen textual ───────────────────────────────────────────────────────
    LABELS = {
        'FUGA_INMINENTE': 'RIESGO DE FUGA — INTERVENCIÓN URGENTE',
        'DETERIORO':      'DETERIORO DE TENDENCIA — VIGILANCIA ACTIVA',
        'VIGILAR':        'SEÑAL DÉBIL NEGATIVA — MONITORIZAR',
        'OK':             'TENDENCIA ESTABLE O CRECIENTE',
    }
    print(f"{'═'*65}")
    print(f"  {ALERT_ICON[alert]}  {LABELS[alert]}")
    print(f"{'═'*65}")
    print(f"  Cliente {client_id}  |  {familia}  |  Método: {method}")
    print(f"  Hoy simulado: {today_date.strftime('%B %Y')}")
    print(f"{'─'*65}")
    if n_hist >= 2:
        print(f"  Histórico de ref.:  {cm_known['year_month_dt'].iloc[0].strftime('%b %Y')} → "
              f"{cm_known['year_month_dt'].iloc[win_start-1].strftime('%b %Y')}  ({n_hist}m)")
    print(f"  Ventana análisis:   {split_date.strftime('%b %Y')} → "
          f"{today_date.strftime('%b %Y')}  ({n_recent}m)")
    print(f"{'─'*65}")
    print(f"  Pendiente histórica: {slope_hist:+.1f} €/mes  ({rel_hist:+.1%}/mes)")
    print(f"  Pendiente reciente:  {slope_rec:+.1f} €/mes  ({rel_rec:+.1%}/mes)")
    print(f"  Significatividad:    p = {p_rec:.3f}  "
          f"({'✓ significativo' if p_rec < 0.15 else '✗ no significativo'})")
    print(f"  Nivel tendencia hoy: {level_now:,.0f} €/mes")
    print(f"{'─'*65}")

    if alert == 'FUGA_INMINENTE':
        print(f"  La tendencia de los últimos {n_recent}m es negativa y significativa,")
        print(f"  y peor que el comportamiento histórico del cliente.")
        print(f"  ACCIÓN A DÍA DE HOY ({today_date.strftime('%b %Y')}): Contactar URGENTEMENTE")
        print(f"  con oferta personalizada. El cliente está derivando compras a la competencia.")
    elif alert == 'DETERIORO':
        print(f"  Tendencia negativa significativa en los últimos {n_recent}m ({slope_rec:+.0f} €/mes).")
        print(f"  ACCIÓN A DÍA DE HOY ({today_date.strftime('%b %Y')}): Revisar la relación")
        print(f"  comercial e identificar la causa del deterioro.")
    elif alert == 'VIGILAR':
        print(f"  Tendencia ligeramente negativa pero no significativa (p={p_rec:.2f}).")
        print(f"  ACCIÓN A DÍA DE HOY ({today_date.strftime('%b %Y')}): Monitorizar.")
        print(f"  Si la tendencia se confirma el próximo mes, activar intervención.")
    else:
        print(f"  Tendencia estable o creciente en los últimos {n_recent}m ({slope_rec:+.0f} €/mes).")
        print(f"  ACCIÓN A DÍA DE HOY ({today_date.strftime('%b %Y')}): Seguimiento estándar.")
    print(f"{'═'*65}")


def update_fuga(change=None):
    with out_fuga:
        clear_output(wait=True)
        if fuga_client_w.value is not None:
            plot_fuga_alert(fuga_client_w.value, fuga_familia_w.value,
                            fuga_dia_avui_w.value, fuga_n_recent_w.value)

fuga_familia_w.observe(update_fuga,   names='value')
fuga_client_w.observe(update_fuga,    names='value')
fuga_dia_avui_w.observe(update_fuga,  names='value')
fuga_n_recent_w.observe(update_fuga,  names='value')

_update_fuga_sliders(fuga_client_w.value, fuga_familia_w.value)

fuga_header = widgets.HTML(
    '<h3 style="margin:4px 0;color:#B71C1C">Alerta de Fuga — Detección de Deterioro</h3>'
    '<p style="color:#555;margin:2px 0">'
    '<b>Slider 1 (Hoy simulado):</b> mueve este punto para situarte en cualquier mes del historial. '
    'El sistema usa solo los datos que habrías tenido <i>hasta ese momento</i>. '
    '<b>Slider 2 (Ventana análisis):</b> cuántos meses recientes examinar para detectar el deterioro. '
    'El resto = histórico de referencia. Sin proyección futura — la alerta es para actuar HOY.</p>'
)
fuga_row1 = widgets.HBox([fuga_familia_w, fuga_client_w],
                          layout=widgets.Layout(gap='10px', align_items='center'))
fuga_row2 = widgets.HBox([fuga_dia_avui_w])
fuga_row3 = widgets.HBox([fuga_n_recent_w])
fuga_row4 = widgets.HBox([fuga_date_label])
display(widgets.VBox([fuga_header, fuga_row1, fuga_row2, fuga_row3, fuga_row4, out_fuga]))

update_fuga()


---

## 7. Explorador Interactivo con Alertas — LightGBM + SHAP

> **Versión ML de la Sección 4**: LightGBM aprende cuándo comprará el cliente usando estacionalidad, tendencia de ventas y variabilidad del ciclo. Las mismas ventanas ±σ y alertas ⚠️/🔴, pero con una predicción más rica que el EWM puro. SHAP explica por qué el modelo adelanta o retrasa la ventana respecto al ciclo típico.

### ¿Qué añade LightGBM respecto al EWM (Sección 4)?

| Señal | EWM | LightGBM |
|-------|-----|----------|
| Historial de gaps | ✅ | ✅ (feature) |
| Estacionalidad (mes / trimestre) | ❌ | ✅ |
| Tendencia de ventas (3/6/12m) | ❌ | ✅ |
| Variabilidad propia del cliente | ✅ parcial | ✅ (ewm_std feature) |
| Explicabilidad individual | ❌ | ✅ SHAP waterfall |

### Interpretación del SHAP waterfall (debajo del timeline)
- 🔴 Barra a la **derecha** → feature que **retrasa** la compra (más días → más riesgo)
- 🔵 Barra a la **izquierda** → feature que **adelanta** la compra (menos días → cliente activo)
- El punto final = predicción ML del nº de días hasta la próxima compra desde el último pedido conocido


In [ ]:
try:
    import lightgbm as lgb
    import shap
    HAS_LGBM = True
    print(f"✅  LightGBM {lgb.__version__}  |  SHAP {shap.__version__}")
except ImportError as e:
    HAS_LGBM = False
    print(f"⚠️  {e}\n    pip install lightgbm shap")

TIMING_COLS = [
    # Historial de gaps (12)
    'n_pedido', 'gap_previo', 'gap_media', 'gap_std', 'coef_var_gaps',
    'gap_min', 'gap_max', 'tendencia_gaps', 'ratio_gap_prev',
    'ratio_gap_ciclo', 'ewm_ciclo', 'ewm_std',
    # Estacionalidad (4)
    'mes_del_pedido', 'trimestre', 'dia_del_mes', 'dia_semana',
    # Importe del pedido (5)
    'euros_pedido', 'euros_pedido_prev', 'ratio_euros_peds',
    'potencial_eur', 'share_pedido',
    # Ventas rolling (8)
    'roll_3m', 'roll_6m', 'roll_12m', 'std_roll_3m', 'std_roll_6m',
    'growth_3vs6', 'growth_6vs12', 'share_12m',
    # Perfil cliente (2)
    'meses_historial', 'prop_meses_activo',
    # ── Variables del dato crudo — no son feature engineering ─────────────────
    'provincia_cod',        # Provincia codificada (geografía)
    'n_prods_pedido',       # Productos distintos en el pedido
    'unidades_pedido',      # Unidades totales del pedido
    'n_productos_cliente',  # SKUs únicos comprados históricamente
    'tiene_tech',           # Compra también técnicos (0/1)
    'tech_euros_ratio',     # Gasto técnico acumulado / potencial commodity
    'n_tech_prods',         # N.º categorías técnicas distintas
]
TIMING_NAMES_ES = [
    '# Pedido (antigüedad)',      'Gap previo (días)',           'Gap medio histórico (días)',
    'Desv. std gaps (días)',      'Coef. variación gaps',
    'Gap mínimo histórico',       'Gap máximo histórico',
    'Tendencia gaps (días/ped)',  'Aceleración (ratio gaps)',
    'Ratio gap / ciclo EWM',      'Ciclo EWM (días)',            'Variabilidad EWM (σ)',
    'Mes del pedido',             'Trimestre',                   'Día del mes',    'Día semana',
    'Importe pedido (€)',         'Importe pedido anterior (€)', 'Ratio importes',
    'Potencial anual (€)',         'Share pedido / potencial',
    'Ventas acum. 3m (€)',        'Ventas acum. 6m (€)',         'Ventas acum. 12m (€)',
    'Volatilidad 3m',             'Volatilidad 6m',
    'Ratio ventas 3m/6m',         'Ratio ventas 6m/12m',         'Share of Wallet 12m',
    'Meses de historial',         'Proporción meses activos',
    # Dato crudo
    'Provincia (cód.)',           'Productos distintos en pedido',
    'Unidades del pedido',        'SKUs únicos del cliente',
    'Cliente técnico (0/1)',      'Ratio gasto técnico / potencial',
    'N.º categorías técnicas',
]

HALF_LIFE_T = 3.0

def _ewm_stats(gaps_arr, half_life=HALF_LIFE_T):
    n = len(gaps_arr)
    if n == 0:
        return np.nan, np.nan
    lam = np.log(2) / max(half_life, 0.1)
    w   = np.exp(lam * np.arange(n))
    w  /= w.sum()
    c   = float(np.dot(w, gaps_arr))
    s   = float(np.sqrt(np.dot(w, (gaps_arr - c) ** 2))) if n > 1 else c * 0.3
    return c, max(s, c * 0.05)


if HAS_LGBM:
    # ── Lookups de dato crudo (globales, usados también en el explorador) ─────
    _all_provs = sorted(comm_base['Provincia'].dropna().unique())
    PROV_MAP   = {p: i for i, p in enumerate(_all_provs)}

    _tech_profile = (
        tech_base.groupby('Id_Cliente')
        .agg(
            _n_tech_ped  = ('Num.Fact',    'nunique'),
            _tech_euros  = ('Valores_H',   'sum'),
            _n_tech_prods= ('Id_Producto', 'nunique'),
        )
        .reset_index()
    )

    _prod_breadth = (
        comm_base.groupby(['Id_Cliente', 'Familia_Potencial'])['Id_Producto']
        .nunique().rename('n_productos_cliente').reset_index()
    )

    # comm_tl enriquecido con columnas del dato crudo
    comm_tl_rich = (
        comm_base
        .groupby(['Id_Cliente', 'Familia_Potencial', 'Num.Fact'])
        .agg(
            Fecha          = ('Fecha',       'min'),
            euros          = ('Valores_H',   'sum'),
            unidades       = ('Unidades',    'sum'),
            n_prods_pedido = ('Id_Producto', 'nunique'),
            Provincia      = ('Provincia',   'first'),
        )
        .reset_index()
        .merge(_tech_profile,  on='Id_Cliente',                          how='left')
        .merge(_prod_breadth,  on=['Id_Cliente', 'Familia_Potencial'],    how='left')
    )
    for col in ['_n_tech_ped', '_tech_euros', '_n_tech_prods']:
        comm_tl_rich[col] = comm_tl_rich[col].fillna(0)
    comm_tl_rich['n_productos_cliente'] = comm_tl_rich['n_productos_cliente'].fillna(1)

    print(f"⏳  Construyendo dataset ({len(TIMING_COLS)} features = 31 derivadas + 7 dato crudo)...")

    def build_timing_dataset(comm_tl_df, monthly_df):
        rows = []
        for (cid, fam), grp in comm_tl_df.groupby(['Id_Cliente', 'Familia_Potencial']):
            grp = grp.sort_values('Fecha').reset_index(drop=True)
            if len(grp) < 3:
                continue
            grp['gap_dies'] = grp['Fecha'].diff().dt.days

            mask_m    = (monthly_df['Id_Cliente'] == cid) & (monthly_df['Familia_Potencial'] == fam)
            mon       = monthly_df[mask_m].set_index('year_month_dt').sort_index()
            potencial = float(mon['potencial_eur'].iloc[0]) if len(mon) > 0 else 0.0
            mon_euros = mon['euros_venuts'] if len(mon) > 0 else pd.Series(dtype=float)
            if len(mon_euros):
                all_m     = pd.date_range(mon_euros.index.min(), REFERENCE_DATE, freq='MS')
                mon_euros = mon_euros.reindex(all_m, fill_value=0.0)

            gaps_all   = grp['gap_dies'].values
            euros_all  = grp['euros'].values
            fechas_all = grp['Fecha'].values
            uds_all    = grp['unidades'].values
            nprods_all = grp['n_prods_pedido'].values
            # Cliente-level raw data (constant within the group)
            prov_cod    = float(PROV_MAP.get(grp['Provincia'].iloc[0], -1))
            n_prods_cli = float(grp['n_productos_cliente'].iloc[0])
            tiene_tech  = float(grp['_n_tech_ped'].iloc[0] > 0)
            tech_euros  = float(grp['_tech_euros'].iloc[0])
            n_tech_p    = float(grp['_n_tech_prods'].iloc[0])
            tech_ratio  = tech_euros / max(potencial, 1.0)

            for i in range(1, len(grp) - 1):
                gap_prev = float(gaps_all[i])
                gap_next = float(gaps_all[i + 1])
                if any(np.isnan(v) or v <= 0 for v in (gap_prev, gap_next)):
                    continue

                prev_gaps = np.array(
                    [g for g in gaps_all[1:i+1] if not np.isnan(g) and g > 0], dtype=float
                )
                if len(prev_gaps) == 0:
                    continue

                ewm_c, ewm_s = _ewm_stats(prev_gaps)
                gap_media    = float(prev_gaps.mean())
                gap_std      = float(prev_gaps.std()) if len(prev_gaps) > 1 else gap_media * 0.3
                cv_gaps      = gap_std / max(gap_media, 1.0)
                tend_gaps    = float(np.polyfit(np.arange(len(prev_gaps)), prev_gaps, 1)[0]) \
                               if len(prev_gaps) >= 3 else 0.0
                ratio_gap_prev = gap_prev / float(prev_gaps[-2]) \
                                 if len(prev_gaps) >= 2 else 1.0

                fecha_i    = pd.Timestamp(fechas_all[i])
                euros_i    = float(euros_all[i])
                euros_prev = float(euros_all[i - 1]) if i > 0 else euros_i

                month_i = fecha_i.to_period('M').to_timestamp()
                r3 = r6 = r12 = std3 = std6 = 0.0
                if len(mon_euros) and month_i in mon_euros.index:
                    idx_m = mon_euros.index.get_loc(month_i)
                    s3  = mon_euros.values[max(0, idx_m-2):idx_m+1]
                    s6  = mon_euros.values[max(0, idx_m-5):idx_m+1]
                    s12 = mon_euros.values[max(0, idx_m-11):idx_m+1]
                    r3   = float(s3.sum());   r6  = float(s6.sum());  r12 = float(s12.sum())
                    std3 = float(s3.std()) if len(s3) > 1 else 0.0
                    std6 = float(s6.std()) if len(s6) > 1 else 0.0

                n_meses_hist = max((fecha_i - pd.Timestamp(fechas_all[0])).days / 30, 1.0)
                prop_activo  = float(i) / n_meses_hist

                rows.append({
                    'Id_Cliente': cid, 'Familia_Potencial': fam, 'Fecha': fecha_i,
                    'n_pedido':          float(i),
                    'gap_previo':        gap_prev,
                    'gap_media':         gap_media,
                    'gap_std':           gap_std,
                    'coef_var_gaps':     cv_gaps,
                    'gap_min':           float(prev_gaps.min()),
                    'gap_max':           float(prev_gaps.max()),
                    'tendencia_gaps':    tend_gaps,
                    'ratio_gap_prev':    ratio_gap_prev,
                    'ratio_gap_ciclo':   gap_prev / max(ewm_c, 1.0),
                    'ewm_ciclo':         ewm_c,
                    'ewm_std':           ewm_s,
                    'mes_del_pedido':    float(fecha_i.month),
                    'trimestre':         float((fecha_i.month - 1) // 3 + 1),
                    'dia_del_mes':       float(fecha_i.day),
                    'dia_semana':        float(fecha_i.dayofweek),
                    'euros_pedido':      euros_i,
                    'euros_pedido_prev': euros_prev,
                    'ratio_euros_peds':  euros_i / max(euros_prev, 1.0),
                    'potencial_eur':     potencial,
                    'share_pedido':      euros_i / max(potencial, 1.0),
                    'roll_3m':           r3,
                    'roll_6m':           r6,
                    'roll_12m':          r12,
                    'std_roll_3m':       std3,
                    'std_roll_6m':       std6,
                    'growth_3vs6':       r3 / max(r6,  1.0),
                    'growth_6vs12':      r6 / max(r12, 1.0),
                    'share_12m':         min(r12 / max(potencial, 1.0), 1.0),
                    'meses_historial':   n_meses_hist,
                    'prop_meses_activo': prop_activo,
                    # Dato crudo
                    'provincia_cod':       prov_cod,
                    'n_prods_pedido':      float(nprods_all[i]),
                    'unidades_pedido':     float(uds_all[i]),
                    'n_productos_cliente': n_prods_cli,
                    'tiene_tech':          tiene_tech,
                    'tech_euros_ratio':    tech_ratio,
                    'n_tech_prods':        n_tech_p,
                    'gap_siguiente':       gap_next,
                })
        return pd.DataFrame(rows)

    timing_df = build_timing_dataset(comm_tl_rich, monthly)
    print(f"✅  {len(timing_df):,} obs  |  {len(TIMING_COLS)} features  |  "
          f"Gap medio: {timing_df['gap_siguiente'].mean():.1f}d ± {timing_df['gap_siguiente'].std():.1f}d")

    cutoff_t   = timing_df['Fecha'].quantile(0.80)
    train_t    = timing_df[timing_df['Fecha'] <= cutoff_t]
    test_t     = timing_df[timing_df['Fecha'] >  cutoff_t]
    X_tr, y_tr = train_t[TIMING_COLS], train_t['gap_siguiente']
    X_te, y_te = test_t[TIMING_COLS],  test_t['gap_siguiente']
    print(f"  Train: {len(X_tr):,}  |  Test: {len(X_te):,}")

    _base_params = dict(
        n_estimators=500, learning_rate=0.04,
        num_leaves=63, min_child_samples=15,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1,
    )
    lgbm_quantiles = {}
    mae_ewm = float(np.mean(np.abs(test_t['ewm_ciclo'].values - y_te.values)))
    print(f"\n  Baseline EWM  MAE = {mae_ewm:.1f} días")

    for q in [0.10, 0.50, 0.90]:
        m = lgb.LGBMRegressor(objective='quantile', alpha=q, **_base_params)
        m.fit(X_tr, y_tr,
              eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])
        lgbm_quantiles[q] = m
        y_p   = m.predict(X_te)
        mae_q = float(np.mean(np.abs(y_p - y_te)))
        cov   = float(np.mean(y_te.values < y_p))
        print(f"  Q{q*100:.0f}: MAE={mae_q:.1f}d  |  cobertura real={cov:.1%} (ideal≈{q:.0%})")

    timing_explainer = shap.TreeExplainer(lgbm_quantiles[0.50])
    sv_timing        = timing_explainer.shap_values(X_te)
    print(f"\n✅  Modelos Q10/Q50/Q90 y SHAP listos  |  {len(TIMING_COLS)} features (7 del dato crudo)")

    shap.summary_plot(sv_timing, X_te.values, feature_names=TIMING_NAMES_ES,
                      plot_type='dot', show=False, max_display=20)
    plt.gcf().set_size_inches(11, 9)
    plt.gcf().axes[0].set_title(
        'Impacto SHAP sobre días hasta la próxima compra — Modelo Q50\n'
        '→ derecha = retrasa la compra · → izquierda = la adelanta',
        fontsize=10
    )
    plt.tight_layout()
    plt.show()

    shap.summary_plot(sv_timing, X_te.values, feature_names=TIMING_NAMES_ES,
                      plot_type='bar', show=False, max_display=20)
    plt.gcf().set_size_inches(8, 8)
    plt.gcf().axes[0].set_title('Importancia global — Timing de compra (Q50) — 38 features', fontsize=10)
    plt.tight_layout()
    plt.show()


In [ ]:
if HAS_LGBM:
    # ── _build_timing_feat: 38 features → (Q10, Q50, Q90, SHAP, feat_vals) ───

    def _build_timing_feat(known_df, familia, primer_date, dia_avui_sim, half_life):
        client_id   = known_df['Id_Cliente'].iloc[0]
        known_sort  = known_df.sort_values('dies_des_del_primer')
        purch_days  = known_sort['dies_des_del_primer'].values.astype(float)

        if len(purch_days) < 2:
            return None, None, None, None, None

        gaps = np.diff(purch_days)
        gaps = gaps[gaps > 0]
        if len(gaps) == 0:
            return None, None, None, None, None

        ewm_c, ewm_s   = _ewm_stats(gaps, half_life)
        gap_prev       = float(gaps[-1])
        gap_media      = float(gaps.mean())
        gap_std        = float(gaps.std()) if len(gaps) > 1 else gap_media * 0.3
        cv_gaps        = gap_std / max(gap_media, 1.0)
        tend_gaps      = float(np.polyfit(np.arange(len(gaps)), gaps, 1)[0]) \
                         if len(gaps) >= 3 else 0.0
        ratio_gap_prev = gap_prev / float(gaps[-2]) if len(gaps) >= 2 else 1.0

        dies_ultim = purch_days[-1]
        last_date  = primer_date + pd.Timedelta(days=int(dies_ultim))
        euros_last = float(known_sort['euros'].iloc[-1])
        euros_prev = float(known_sort['euros'].iloc[-2]) if len(known_sort) > 1 else euros_last

        mask_m    = (monthly['Id_Cliente'] == client_id) & (monthly['Familia_Potencial'] == familia)
        mon_data  = monthly[mask_m]
        potencial = float(mon_data['potencial_eur'].iloc[0]) if len(mon_data) > 0 else 0.0
        mon_euros = mon_data.set_index('year_month_dt')['euros_venuts'].sort_index() \
                    if len(mon_data) > 0 else pd.Series(dtype=float)
        r3 = r6 = r12 = std3 = std6 = 0.0
        n_meses_hist = max((last_date - primer_date).days / 30, 1.0)
        if len(mon_euros):
            all_m     = pd.date_range(mon_euros.index.min(), REFERENCE_DATE, freq='MS')
            mon_euros = mon_euros.reindex(all_m, fill_value=0.0)
            mkey = last_date.to_period('M').to_timestamp()
            if mkey in mon_euros.index:
                idx_m = mon_euros.index.get_loc(mkey)
                s3  = mon_euros.values[max(0, idx_m-2):idx_m+1]
                s6  = mon_euros.values[max(0, idx_m-5):idx_m+1]
                s12 = mon_euros.values[max(0, idx_m-11):idx_m+1]
                r3   = float(s3.sum()); r6 = float(s6.sum()); r12 = float(s12.sum())
                std3 = float(s3.std()) if len(s3) > 1 else 0.0
                std6 = float(s6.std()) if len(s6) > 1 else 0.0

        prop_activo = float(len(purch_days) - 1) / n_meses_hist

        # Dato crudo: lookup desde comm_tl_rich (global)
        mask_r      = (comm_tl_rich['Id_Cliente'] == client_id) & \
                      (comm_tl_rich['Familia_Potencial'] == familia)
        rich        = comm_tl_rich[mask_r].sort_values('Fecha')
        today_date  = primer_date + pd.Timedelta(days=int(dia_avui_sim))
        rich_known  = rich[rich['Fecha'] <= today_date]
        prov_cod    = float(PROV_MAP.get(rich['Provincia'].iloc[0], -1))     if len(rich) > 0 else -1.0
        n_prods_ped = float(rich_known['n_prods_pedido'].iloc[-1])          if len(rich_known) > 0 else 1.0
        uds_ped     = float(rich_known['unidades'].iloc[-1])                if len(rich_known) > 0 else 0.0
        n_prods_cli = float(rich['n_productos_cliente'].iloc[0])             if len(rich) > 0 else 1.0
        tiene_tech  = float(rich['_n_tech_ped'].iloc[0] > 0)                if len(rich) > 0 else 0.0
        tech_euros  = float(rich['_tech_euros'].iloc[0])                    if len(rich) > 0 else 0.0
        n_tech_p    = float(rich['_n_tech_prods'].iloc[0])                  if len(rich) > 0 else 0.0
        tech_ratio  = tech_euros / max(potencial, 1.0)

        feat = pd.DataFrame([{
            'n_pedido':          float(len(purch_days) - 1),
            'gap_previo':        gap_prev,
            'gap_media':         gap_media,
            'gap_std':           gap_std,
            'coef_var_gaps':     cv_gaps,
            'gap_min':           float(gaps.min()),
            'gap_max':           float(gaps.max()),
            'tendencia_gaps':    tend_gaps,
            'ratio_gap_prev':    ratio_gap_prev,
            'ratio_gap_ciclo':   gap_prev / max(ewm_c, 1.0),
            'ewm_ciclo':         ewm_c,
            'ewm_std':           ewm_s,
            'mes_del_pedido':    float(last_date.month),
            'trimestre':         float((last_date.month - 1) // 3 + 1),
            'dia_del_mes':       float(last_date.day),
            'dia_semana':        float(last_date.dayofweek),
            'euros_pedido':      euros_last,
            'euros_pedido_prev': euros_prev,
            'ratio_euros_peds':  euros_last / max(euros_prev, 1.0),
            'potencial_eur':     potencial,
            'share_pedido':      euros_last / max(potencial, 1.0),
            'roll_3m':           r3,
            'roll_6m':           r6,
            'roll_12m':          r12,
            'std_roll_3m':       std3,
            'std_roll_6m':       std6,
            'growth_3vs6':       r3 / max(r6,  1.0),
            'growth_6vs12':      r6 / max(r12, 1.0),
            'share_12m':         min(r12 / max(potencial, 1.0), 1.0),
            'meses_historial':   n_meses_hist,
            'prop_meses_activo': prop_activo,
            'provincia_cod':     prov_cod,
            'n_prods_pedido':    n_prods_ped,
            'unidades_pedido':   uds_ped,
            'n_productos_cliente': n_prods_cli,
            'tiene_tech':        tiene_tech,
            'tech_euros_ratio':  tech_ratio,
            'n_tech_prods':      n_tech_p,
        }], columns=TIMING_COLS)

        q10 = max(1.0, float(lgbm_quantiles[0.10].predict(feat)[0]))
        q50 = max(q10 + 1.0, float(lgbm_quantiles[0.50].predict(feat)[0]))
        q90 = max(q50 + 1.0, float(lgbm_quantiles[0.90].predict(feat)[0]))

        sv_row = timing_explainer.shap_values(feat)
        if isinstance(sv_row, list):
            sv_row = sv_row[0]
        return q10, q50, q90, sv_row[0], feat.values[0]

    # ── Widgets (misma estructura que Sección 4) ──────────────────────────────
    lgbm_familia_w = widgets.Dropdown(
        options=FAMILIES, value='Anestesia',
        description='Familia:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='240px')
    )
    lgbm_client_w = widgets.Dropdown(
        options=get_clients_for_familia('Anestesia'),
        description='Cliente ID:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='260px')
    )
    lgbm_minped_w = widgets.IntSlider(
        value=3, min=1, max=10, step=1,
        description='Min pedidos (establecido):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='440px')
    )
    lgbm_avui_w = widgets.IntSlider(
        value=1000, min=1, max=2000, step=5,
        description='Hoy simulado (días):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='600px')
    )
    lgbm_hl_w = widgets.FloatSlider(
        value=3.0, min=0.5, max=10.0, step=0.5,
        description='Vida media EWM (ref):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='440px'),
        readout_format='.1f'
    )
    out_lgbm = widgets.Output()

    def _update_lgbm_slider(client_id, familia):
        cdata = _get_client_timeline(client_id, familia)
        if len(cdata) == 0:
            return
        primer_date    = cdata['primer_pedido'].iloc[0]
        dies_avui_real = (REFERENCE_DATE - primer_date).days
        first_gap = int(cdata['dies_des_del_primer'].iloc[1]) if len(cdata) > 1 else 10
        lgbm_avui_w.min   = max(first_gap, 1)
        lgbm_avui_w.max   = dies_avui_real
        lgbm_avui_w.value = max(int(dies_avui_real * 0.60), lgbm_avui_w.min)

    def on_lgbm_familia(change):
        nc = get_clients_for_familia(change['new'])
        lgbm_client_w.options = nc
        if nc:
            lgbm_client_w.value = nc[0]
            _update_lgbm_slider(nc[0], change['new'])

    def on_lgbm_client(change):
        _update_lgbm_slider(change['new'], lgbm_familia_w.value)

    lgbm_familia_w.observe(on_lgbm_familia, names='value')
    lgbm_client_w.observe(on_lgbm_client,   names='value')

    # ── Función de plot ───────────────────────────────────────────────────────

    def plot_lgbm_explorer(client_id, familia, min_pedidos, dia_avui_sim, half_life):
        cdata = _get_client_timeline(client_id, familia)
        if len(cdata) == 0:
            print(f'Sin datos para {client_id} | {familia}')
            return

        primer_date = cdata['primer_pedido'].iloc[0]
        known  = cdata[cdata['dies_des_del_primer'] <= dia_avui_sim]
        if len(known) == 0:
            print('Desplaza el slider a la derecha.')
            return

        dies_ultim = int(known['dies_des_del_primer'].max())
        es_nou     = len(known) < min_pedidos

        cicle_ewm, cicle_std, norm_w, cicle_tipus, bar_color, _ = _get_cycle(
            known, familia, es_nou, half_life
        )

        # LightGBM Q10/Q50/Q90
        q10_gap = q50_gap = q90_gap = cicle_ewm
        lgbm_sv = lgbm_xv = None
        if not es_nou:
            res = _build_timing_feat(known, familia, primer_date, dia_avui_sim, half_life)
            if res[0] is not None:
                q10_gap, q50_gap, q90_gap, lgbm_sv, lgbm_xv = res

        # Ventanas Q-based para hasta 3 ciclos
        # Ciclo k: verde [ultim + (k-1)*Q50 + Q10, ultim + k*Q50]
        #          naranja [ultim + k*Q50, ultim + (k-1)*Q50 + Q90]
        #          rojo (crítico) [ultim + (k-1)*Q50 + Q90, + 0.5*(Q90-Q50)]
        preds_lgbm = []
        extra = (q90_gap - q50_gap) * 0.5
        for k in range(1, 4):
            low  = dies_ultim + (k - 1) * q50_gap + q10_gap
            mid  = dies_ultim + k * q50_gap
            high = dies_ultim + (k - 1) * q50_gap + q90_gap
            risk = high + extra
            if low > dies_ultim + 4 * q50_gap:
                break
            preds_lgbm.append({'low': low, 'mid': mid, 'high': high, 'risk_high': risk})

        # EWM referencia (primera ventana ±0.5σ)
        ewm_ref = {
            'low':  dies_ultim + cicle_ewm - 0.5 * cicle_std,
            'day':  dies_ultim + cicle_ewm,
            'high': dies_ultim + cicle_ewm + 0.5 * cicle_std,
        }

        # Alertas
        anticipation_alerts, cierre_alerts, critical_alerts = [], [], []
        for p in preds_lgbm:
            orders_in = known[(known['dies_des_del_primer'] >= p['low']) &
                              (known['dies_des_del_primer'] <= p['high'])]
            has_buy = len(orders_in) > 0
            if p['low'] <= dia_avui_sim <= p['mid'] and not has_buy:
                anticipation_alerts.append(p)
            elif p['mid'] < dia_avui_sim <= p['high'] and not has_buy:
                cierre_alerts.append(p)
            elif p['high'] < dia_avui_sim <= p['risk_high'] and not has_buy:
                critical_alerts.append(p)

        gap_positions = known['dies_des_del_primer'].values[1:]

        # ── Figura 1: Timeline ────────────────────────────────────────────────
        fig, (ax, ax_w) = plt.subplots(
            2, 1, figsize=(15, 6.5),
            gridspec_kw={'height_ratios': [5, 1.2], 'hspace': 0.06}
        )
        ymax  = float(cdata['euros'].max()) if cdata['euros'].max() > 0 else 1.0
        bar_w = max(cicle_ewm * 0.05, 3)

        # EWM ref (punteado azul claro)
        ax.axvspan(ewm_ref['low'], ewm_ref['high'], alpha=0.10, color='#1565C0', zorder=1)
        ax.axvline(ewm_ref['day'], color='#1565C0', ls=':', lw=1.2, alpha=0.6, zorder=2)

        # LightGBM ventanas Q-based
        for p in preds_lgbm:
            ax.axvspan(p['low'],  p['mid'],       alpha=0.18, color='#2E7D32', zorder=1)  # verde Q10→Q50
            ax.axvspan(p['mid'],  p['high'],      alpha=0.12, color='#F57F17', zorder=1)  # naranja Q50→Q90
            ax.axvspan(p['high'], p['risk_high'], alpha=0.08, color='#B71C1C', zorder=1)  # rojo >Q90
            ax.axvline(p['mid'],  color='#2E7D32', ls='--', lw=0.9, alpha=0.55, zorder=2)

        # Alertas
        for i, p in enumerate(anticipation_alerts):
            ax.axvline(p['low'], color='#FF6F00', lw=3, zorder=8, alpha=0.9)
            y_ann = ymax * (0.82 - i * 0.12)
            ax.annotate(
                '⚠️ INICIO VENTANA\nCONTACTAR AHORA',
                xy=(p['low'], y_ann * 0.88),
                xytext=(p['low'] + q50_gap * 0.15, y_ann),
                fontsize=7.5, color='#BF360C', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#FF6F00', lw=1.5),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0',
                          edgecolor='#FF6F00', lw=1.5), zorder=9
            )
        for i, p in enumerate(cierre_alerts):
            ax.axvline(p['mid'], color='#E65100', lw=3, zorder=8, alpha=0.9)
            y_ann = ymax * (0.70 - i * 0.12)
            ax.annotate(
                '⚠️ PASADA MEDIANA\nURGENTE — AÚN EN VENTANA',
                xy=(p['mid'], y_ann * 0.88),
                xytext=(p['mid'] + q50_gap * 0.12, y_ann),
                fontsize=7.5, color='#E65100', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.5),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF8E1',
                          edgecolor='#E65100', lw=1.5), zorder=9
            )
        for i, p in enumerate(critical_alerts):
            ax.axvline(p['high'], color='#B71C1C', lw=3, zorder=8, alpha=0.9)
            y_ann = ymax * (0.58 - i * 0.12)
            ax.annotate(
                '🔴 FIN VENTANA — ZONA CRÍTICA\n>90% YA COMPRARON. ¡URGENTE!',
                xy=(p['high'], y_ann * 0.88),
                xytext=(p['high'] + q50_gap * 0.12, y_ann),
                fontsize=7.5, color='#B71C1C', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#B71C1C', lw=1.5),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFEBEE',
                          edgecolor='#B71C1C', lw=1.5), zorder=9
            )

        # Barras historial
        for idx, (_, row) in enumerate(known.iterrows()):
            d = row['dies_des_del_primer']
            if idx > 0 and len(norm_w) > 0 and (idx - 1) < len(norm_w):
                w_min, w_max = norm_w.min(), norm_w.max()
                alpha_bar = 0.28 + 0.67 * (norm_w[idx-1] - w_min) / max(w_max - w_min, 1e-9)
            else:
                alpha_bar = 0.40
            ax.bar(d, row['euros'], width=bar_w, color=bar_color, alpha=alpha_bar, zorder=3)
            if row['euros'] > 0:
                ax.text(d, row['euros'] + ymax * 0.015, f"{row['euros']:.0f}",
                        ha='center', va='bottom', fontsize=7.5, color='#222')

        ax.axvline(dia_avui_sim, color='black', lw=2.4, zorder=6)

        diff_txt = ''
        if lgbm_sv is not None:
            diff_txt = f'   ML ajusta {q50_gap - cicle_ewm:+.0f}d vs EWM'
        estat = 'NUEVO' if es_nou else 'ESTABLECIDO'
        ax.set_title(
            f'Cliente {client_id}  |  {familia}  |  {estat}  |  '
            f'EWM ref: {cicle_ewm:.0f}d (🔵)  |  '
            f'LightGBM — Q10:{q10_gap:.0f}d  Q50:{q50_gap:.0f}d  Q90:{q90_gap:.0f}d{diff_txt}',
            fontsize=9.5, pad=8
        )
        ax.set_ylabel('Importe (€)', fontsize=10)
        handles = [
            mpatches.Patch(color='#1565C0', alpha=0.20,
                           label=f'EWM referencia ({cicle_ewm:.0f}d ±0.5σ)'),
            mpatches.Patch(color='#2E7D32', alpha=0.30,
                           label=f'Ventana Q10→Q50 ({q10_gap:.0f}→{q50_gap:.0f}d) — zona probable'),
            mpatches.Patch(color='#F57F17', alpha=0.25,
                           label=f'Ventana Q50→Q90 ({q50_gap:.0f}→{q90_gap:.0f}d) — zona tardía'),
            mpatches.Patch(color='#B71C1C', alpha=0.20,
                           label=f'Zona crítica >Q90 ({q90_gap:.0f}d)'),
            plt.Line2D([0],[0], color='black', lw=2,
                       label=f'Hoy simulado (día {dia_avui_sim})'),
            plt.Line2D([0],[0], color='#FF6F00', lw=2.5,
                       label='⚠️ Alerta Anticipación'),
            plt.Line2D([0],[0], color='#B71C1C', lw=2.5,
                       label='🔴 Alerta Crítica (>Q90)'),
        ]
        ax.legend(handles=handles, loc='upper left', fontsize=7.5)
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticklabels([])
        x_right = max(dia_avui_sim * 1.05,
                      dies_ultim + 3.5 * q50_gap,
                      cdata['dies_des_del_primer'].max() * 1.03)
        ax.set_xlim(-bar_w * 2, x_right)
        ax.set_ylim(0, ymax * 1.22)

        if len(norm_w) > 0:
            cmap_vals    = 0.3 + 0.7 * norm_w / norm_w.max()
            bar_colors_w = plt.cm.Blues(cmap_vals)
            ax_w.bar(gap_positions, norm_w * 100, width=bar_w * 1.5,
                     color=bar_colors_w, alpha=0.9)
            for gp, gw in zip(gap_positions, norm_w):
                ax_w.text(gp, gw * 100 + norm_w.max() * 2, f'{gw*100:.1f}%',
                          ha='center', va='bottom', fontsize=7, color='#333')
            ax_w.axvline(dia_avui_sim, color='black', lw=1.5, alpha=0.4)
            ax_w.set_xlim(ax.get_xlim())
            ax_w.set_ylim(0, norm_w.max() * 140)
            ax_w.set_ylabel('Peso EWM\n(%)', fontsize=7.5)
            ax_w.grid(axis='y', alpha=0.2)
            ax_w.tick_params(labelsize=7)
        else:
            ax_w.axis('off')
        ax_w.set_xlabel('Días desde el primer pedido  (día 0 = primera compra)', fontsize=10)
        plt.tight_layout()
        plt.show()

        # ── Figura 2: SHAP waterfall (Q50) ────────────────────────────────────
        if lgbm_sv is not None:
            base_v = float(timing_explainer.expected_value
                           if not hasattr(timing_explainer.expected_value, '__len__')
                           else timing_explainer.expected_value[0])
            exp_obj = shap.Explanation(
                values=lgbm_sv,
                base_values=base_v,
                data=lgbm_xv,
                feature_names=TIMING_NAMES_ES
            )
            try:
                shap.plots.waterfall(exp_obj, max_display=12, show=False)
            except AttributeError:
                shap.waterfall_plot(base_v, lgbm_sv,
                                    pd.Series(lgbm_xv, index=TIMING_NAMES_ES),
                                    max_display=12)
            plt.gcf().set_size_inches(10, 6)
            plt.gcf().suptitle(
                f'SHAP — ¿Por qué LightGBM predice Q50={q50_gap:.0f}d? '
                f'(EWM ref: {cicle_ewm:.0f}d  |  ajuste: {q50_gap - cicle_ewm:+.0f}d)\n'
                '→ derecha = retrasa compra  ·  → izquierda = adelanta compra',
                fontsize=9.5, y=1.01
            )
            plt.tight_layout()
            plt.show()

        # ── Resumen texto ─────────────────────────────────────────────────────
        print(f"{'─'*64}")
        print(f"  Cliente {client_id}  |  {familia}  |  Hoy: día {dia_avui_sim}  "
              f"({(primer_date + pd.Timedelta(days=dia_avui_sim)).strftime('%d/%m/%Y')})")
        print(f"  EWM ciclo ref.:      {cicle_ewm:.0f}d ± {cicle_std:.0f}d")
        print(f"  LightGBM Q10/Q50/Q90: {q10_gap:.0f}d / {q50_gap:.0f}d / {q90_gap:.0f}d")
        print(f"  Ventana probable:    {q10_gap:.0f}d → {q50_gap:.0f}d desde el último pedido")
        print(f"  Ventana tardía:      {q50_gap:.0f}d → {q90_gap:.0f}d")
        print(f"  Zona crítica:        >{q90_gap:.0f}d  (90% de clientes similares ya compraron)")
        print(f"{'─'*64}")

        all_alerts = anticipation_alerts + cierre_alerts + critical_alerts
        if all_alerts:
            print(f"{'═'*64}")
            print(f"  🚨 ALERTAS ACTIVAS")
            print(f"{'═'*64}")
            for p in anticipation_alerts:
                d_ini = (primer_date + pd.Timedelta(days=int(p['low']))).strftime('%d/%m/%Y')
                d_mid = (primer_date + pd.Timedelta(days=int(p['mid']))).strftime('%d/%m/%Y')
                print(f"  ⚠️  ANTICIPACIÓN — ventana verde Q10→Q50: {d_ini} → {d_mid}")
                print(f"     Cliente en su ventana habitual. ACCIÓN: Contactar AHORA.")
            for p in cierre_alerts:
                d_mid = (primer_date + pd.Timedelta(days=int(p['mid']))).strftime('%d/%m/%Y')
                d_hi  = (primer_date + pd.Timedelta(days=int(p['high']))).strftime('%d/%m/%Y')
                print(f"  ⚠️  TARDÍO — pasada mediana Q50 ({d_mid}), aún en ventana Q90 ({d_hi})")
                print(f"     ACCIÓN: Llamada urgente antes de que compre a competencia.")
            for p in critical_alerts:
                d_hi  = (primer_date + pd.Timedelta(days=int(p['high']))).strftime('%d/%m/%Y')
                print(f"  🔴 CRÍTICO — superado Q90 ({d_hi}). >90% de clientes similares ya compraron.")
                print(f"     ACCIÓN: Llamada INMEDIATA. Alta prob. de pedido a competencia.")
            print(f"{'═'*64}")
        else:
            print(f"  ✅ Sin alertas activas en el momento simulado.")
            print(f"{'═'*64}")

    def update_lgbm_chart(change=None):
        with out_lgbm:
            clear_output(wait=True)
            if lgbm_client_w.value is not None:
                plot_lgbm_explorer(
                    lgbm_client_w.value, lgbm_familia_w.value,
                    lgbm_minped_w.value, lgbm_avui_w.value,
                    lgbm_hl_w.value
                )

    lgbm_familia_w.observe(update_lgbm_chart, names='value')
    lgbm_client_w.observe(update_lgbm_chart,  names='value')
    lgbm_minped_w.observe(update_lgbm_chart,  names='value')
    lgbm_avui_w.observe(update_lgbm_chart,    names='value')
    lgbm_hl_w.observe(update_lgbm_chart,      names='value')

    lgbm_header = widgets.HTML(
        '<h3 style="margin:4px 0;color:#2E7D32">'
        '7. Explorador LightGBM + SHAP — Alertas con Bandas de Cuantiles</h3>'
        '<p style="color:#555;margin:2px 0">'
        '🔵 <b>EWM punteado</b> = referencia histórica personal. '
        '🟢 <b>Verde Q10→Q50</b> = ventana probable (el 50% ya habrá comprado). '
        '🟡 <b>Naranja Q50→Q90</b> = zona tardía. '
        '🔴 <b>Rojo >Q90</b> = zona crítica (90% similares ya compraron). '
        'El modelo usa 38 features: 31 derivadas del historial + '
        '<b>7 del dato crudo</b> (provincia, productos del pedido, unidades, '
        'SKUs del cliente, si compra técnicos, ratio gasto técnico/potencial, '
        'N.º categorías técnicas). SHAP explica el ajuste sobre cada pedido.</p>'
    )
    lgbm_row1 = widgets.HBox([lgbm_familia_w, lgbm_client_w],
                               layout=widgets.Layout(gap='10px', align_items='center'))
    lgbm_row2 = widgets.HBox([lgbm_minped_w, lgbm_hl_w],
                               layout=widgets.Layout(gap='10px', align_items='center'))
    lgbm_row3 = widgets.HBox([lgbm_avui_w])
    display(widgets.VBox([lgbm_header, lgbm_row1, lgbm_row2, lgbm_row3, out_lgbm]))

    _update_lgbm_slider(lgbm_client_w.value, lgbm_familia_w.value)
    update_lgbm_chart()
